# Use Kaggle as Your Own Remote LLM Server

**Run an open-weight language model on a Kaggle GPU and access it remotely through an HTTPS API.**

This notebook is a practical, step-by-step guide to turning a Kaggle GPU session into a temporary LLM inference server. The model runs inside Kaggle using **Ollama**. A **Cloudflare Quick Tunnel** exposes the Ollama API through a temporary public HTTPS address, so your local computer, Python scripts, AI agents, and compatible applications can send requests to it.

You do not need a powerful GPU on your personal computer. Your computer sends the prompt over the internet; the Kaggle machine performs inference and sends the generated response back.

> **Important:** This is an experimental, temporary server—not free permanent hosting. The Kaggle session must remain active, the public URL is temporary, and the tunnel in this notebook has no API authentication. Never share the URL publicly or send sensitive data through it.

## What you will build

By running the notebook from top to bottom, you will:

1. Check that a Kaggle GPU is available and install the required runtime tools.
2. Start **Ollama**, which serves the model through an HTTP API.
3. Download and configure the reference **Qwen3.8 27B Uncensored GGUF (Q4_K_M)** model.
4. Load the model and test inference inside Kaggle.
5. Start a **Cloudflare Quick Tunnel** to make the Ollama API reachable through HTTPS.
6. Call the model from another computer using either Ollama's native API or an OpenAI-compatible client.
7. Learn where this setup is useful, how to connect applications, and what its limitations are.

## What is an LLM inference server?

A language model normally needs to be loaded into memory before it can generate text. An *inference server* keeps the model running and accepts requests from client applications.

For example, a client sends a message such as “Explain how a transformer works.” The server passes that message to the loaded model, waits for generation, and returns the response as JSON over HTTP.

In this notebook:

- **Kaggle** provides the temporary notebook runtime, GPU (when enabled), memory, and internet access.
- **Ollama** downloads, loads, and serves the model. It exposes endpoints such as `/api/chat` and `/api/generate`.
- **The GGUF model** contains the model weights in a quantized format. Quantization reduces storage and memory requirements compared with full-precision weights, with possible quality/performance trade-offs.
- **Cloudflare Quick Tunnel** creates a temporary HTTPS route from a public `trycloudflare.com` address to the Ollama service running inside the Kaggle session.
- **Your client** can be a Python script, backend, agent, or another application that can send HTTP requests.

## Architecture

```mermaid
flowchart LR
    A["Your local computer / app"] -->|"HTTPS request"| B["Cloudflare Quick Tunnel"]
    B -->|"forwards traffic"| C["Kaggle notebook runtime"]
    C --> D["Ollama API :11434"]
    D --> E["Loaded Qwen GGUF model"]
    E --> D
    D -->|"JSON response"| C
    C -->|"HTTPS response"| B
    B --> A
```

The public URL points to the tunnel, not directly to a GPU. The tunnel forwards HTTP requests to Ollama's local address (`127.0.0.1:11434`) inside Kaggle. Ollama then uses the loaded model to generate the response.

## Before you start: Kaggle settings

Open the notebook's **Settings / Notebook options** and configure:

| Setting | Recommended value |
|---|---|
| Accelerator | GPU — preferably **T4 x2**, if available |
| Internet | On |
| Persistence | Optional; it does not make the running server permanent |

### Hardware and model note

The reference model download is approximately **16.8 GB** before runtime, context, and cache overhead. A single 16 GB T4 cannot keep the full model entirely in VRAM. Ollama may distribute work across two GPUs or offload some layers to system RAM. Whether it fits—and how quickly it runs—depends on the GPU allocation, available system RAM, context length, and runtime conditions.

The reference configuration uses a context size of 65,536 tokens. This is a substantial memory requirement. If the model does not load, reduce the model/context requirements or use a smaller model in a separately adapted version of the notebook. This tutorial preserves the supplied reference implementation rather than changing its model settings.

## How to use this notebook

1. Open this notebook in Kaggle and enable **GPU** and **Internet**.
2. Run the cells in order, from the installation section through the local inference test.
3. Run the Cloudflare Tunnel cell when you are ready to access the server remotely.
4. Copy the printed public URL. Keep it private.
5. On your own computer, run one of the client examples in the sections below. Replace the placeholder tunnel URL with the URL printed by your notebook.
6. Keep the Kaggle session and tunnel running for as long as you need the server.

**Execution location matters:** cells marked as Kaggle notebook cells run in Kaggle. The client examples are intended to be copied into a separate local Python file or terminal environment; they are not meant to be run as Kaggle cells.


## 1. Install the runtime dependencies

**Run this cell in Kaggle.**

This setup cell checks for an NVIDIA GPU and installs the tools used by the rest of the notebook:

- **Ollama:** runs the model and provides the local HTTP API.
- **cloudflared:** creates the Cloudflare Quick Tunnel used later.
- **zstd:** supports handling compressed model downloads where needed.

If the GPU check fails, stop here, enable a GPU accelerator in Kaggle Settings, restart the session, and run this cell again.

The installation is performed inside the temporary Kaggle runtime. It does not install Ollama on your personal computer.

In [1]:
import os
import shutil
import subprocess

def run(command):
    return subprocess.run(command, check=True)

if not shutil.which("nvidia-smi"):
    raise RuntimeError(
        "No NVIDIA GPU detected. Enable a GPU accelerator in Kaggle "
        "Settings, restart the session, and run again."
    )

run(["nvidia-smi"])

if not shutil.which("zstd"):
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "zstd"])

if not shutil.which("ollama"):
    run([
        "bash", "-lc",
        "curl -fsSL https://ollama.com/install.sh | sh"
    ])

if not shutil.which("cloudflared"):
    cloudflared_deb = "/tmp/cloudflared-linux-amd64.deb"
    run([
        "curl", "-fL",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        "-o", cloudflared_deb,
    ])
    run(["dpkg", "-i", cloudflared_deb])

print(subprocess.check_output(["ollama", "--version"], text=True).strip())
print(subprocess.check_output(["cloudflared", "--version"], text=True).strip())

Thu Nov 12 18:38:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


Selecting previously unselected package zstd.
(Reading database ... 130169 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0% 43.9%############################                                         46.8%       57.7%                   67.1%############                         69.1% 82.4%#####################################               82.6%###############################################################       93.9% 99.8%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 19.0M  100 19.0M    0     0  38.6M      0 --:--:-- --:--:-- --:--:-- 38.6M


Selecting previously unselected package cloudflared.
(Reading database ... 130190 files and directories currently installed.)
Preparing to unpack .../cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.11.0) ...
Setting up cloudflared (2026.11.0) ...
Processing triggers for man-db (2.12.0-4build2) ...
cloudflared version 2026.11.0 (built 2026-11-12-12:17 UTC)


## 2. Start Ollama and configure the model

**Run this cell in Kaggle.**

Ollama is started as a background service and listens on `127.0.0.1:11434` inside the Kaggle runtime. This is the private, local address that the later Cloudflare Tunnel forwards requests to.

Important configuration values in this reference setup:

- `OLLAMA_KEEP_ALIVE=-1` asks Ollama to keep the model loaded for the lifetime of the running server.
- `OLLAMA_MODELS` stores model files under `/kaggle/working/ollama-models`.
- `NUM_CTX` sets the configured context window to 65,536 tokens.
- `DRAFT_TOKENS` sets the draft-token parameter used by this MTP model configuration.

The model and context require substantial memory. Do not assume the same settings will work on every Kaggle GPU allocation. The cell also defines a small request helper that the later cells use to communicate with Ollama.

In [2]:
import json
import time
import urllib.request
from pathlib import Path

SOURCE_MODEL = (
    "hf.co/JonathanColetti/"
    "Qwen3.8-27B-Uncensored-GGUF:Q4_K_M"
)
MTP_MODEL = "qwen3.8-27b-uncensored-mtp"
OLLAMA_URL = "http://127.0.0.1:11434"
NUM_CTX = 65536
DRAFT_TOKENS = 2

def ollama_request(endpoint, payload, base_url=OLLAMA_URL, timeout=3600):
    request = urllib.request.Request(
        f"{base_url.rstrip('/')}{endpoint}",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read())

def ollama_is_ready():
    try:
        urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=2)
        return True
    except Exception:
        return False


PERSISTENT_MODEL_DIR = Path(
    "/kaggle/working/ollama-models"
)

PERSISTENT_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

os.environ["OLLAMA_MODELS"] = str(
    PERSISTENT_MODEL_DIR
)


ollama_env = os.environ.copy()
ollama_env.update({
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_KEEP_ALIVE": "-1",
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": "q8_0",
    "OLLAMA_CONTEXT_LENGTH": str(NUM_CTX),
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "1",
})

if not ollama_is_ready():
    ollama_log_path = Path("/tmp/ollama-server.log")
    ollama_log = ollama_log_path.open("a")
    ollama_server_process = subprocess.Popen(
        ["ollama", "serve"],
        env=ollama_env,
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
    )

    for _ in range(60):
        if ollama_is_ready():
            break
        if ollama_server_process.poll() is not None:
            raise RuntimeError(f"Ollama exited early. Check {ollama_log_path}")
        time.sleep(1)
    else:
        raise RuntimeError(f"Ollama did not become ready. Check {ollama_log_path}")

print(f"Ollama ready at {OLLAMA_URL}")

Ollama ready at http://127.0.0.1:11434


## 3. Download, create, and load the model

**Run this cell in Kaggle.**

This cell performs three operations:

1. Downloads the source GGUF model from the model repository configured above.
2. Creates an Ollama model entry named `qwen3.8-27b-uncensored-mtp` and applies the MTP-related parameters.
3. Sends an initial request so Ollama loads the model and keeps it available for subsequent requests.

**The download is large and may take several minutes.** The model files are stored in the current Kaggle session's working storage. A new or reset session may require downloading them again.

Wait for the cell to finish successfully before moving to the inference test.

In [3]:
subprocess.run(
    ["ollama", "pull", SOURCE_MODEL],
    env=ollama_env,
    check=True,
)

ollama_request(
    "/api/create",
    {
        "model": MTP_MODEL,
        "from": SOURCE_MODEL,
        "parameters": {
            "draft_num_predict": DRAFT_TOKENS,
            "num_ctx": NUM_CTX,
        },
        "stream": False,
    },
)

ollama_request(
    "/api/generate",
    {
        "model": MTP_MODEL,
        "prompt": "",
        "stream": False,
        "keep_alive": -1,
        "options": {
            "draft_num_predict": DRAFT_TOKENS,
            "num_ctx": NUM_CTX,
        },
    },
)

print(f"{MTP_MODEL} loaded with MTP and pinned for this session.")
subprocess.run(["ollama", "ps"], check=True)

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏  26 KB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏ 1.1 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏ 5.4 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏  54 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   1% ▕                  ▏ 122 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   1% ▕                  ▏ 153 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   1% ▕                  ▏ 222 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   2% ▕                  ▏ 288 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   2% ▕                  ▏ 317 MB/ 16 GB            

qwen3.8-27b-uncensored-mtp loaded with MTP and pinned for this session.
NAME                                 ID              SIZE     PROCESSOR    CONTEXT    RUNNER      UNTIL   
qwen3.8-27b-uncensored-mtp:latest    0cb38e1ce65b    19 GB    100% GPU     65536      llamacpp    Forever    


CompletedProcess(args=['ollama', 'ps'], returncode=0)

## 4. Test inference inside Kaggle

**Run this cell in Kaggle before starting the tunnel.**

This sends a small test prompt directly to the local Ollama API. The cell prints the generated response and estimates generation speed using the timing fields returned by Ollama.

A successful `READY` response confirms that the model can generate text in the current session. The `ollama ps` output lets you inspect the model currently loaded by Ollama.

If this test fails, troubleshoot the local Ollama/model setup before exposing the API to the internet.

In [4]:
result = ollama_request(
    "/api/generate",
    {
        "model": MTP_MODEL,
        "prompt": "Reply with the single word READY.",
        "think": False,
        "stream": False,
        "keep_alive": -1,
        "options": {
            "draft_num_predict": DRAFT_TOKENS,
            "num_ctx": NUM_CTX,
            "temperature": 0,
        },
    },
)

eval_count = result.get("eval_count", 0)
eval_seconds = result.get("eval_duration", 0) / 1_000_000_000
tokens_per_second = eval_count / eval_seconds if eval_seconds else 0

print("Response:", result.get("response", "").strip())
print(f"Generation speed: {tokens_per_second:.1f} tokens/second")
subprocess.run(["ollama", "ps"], check=True)

Response: READY
Generation speed: 0.1 tokens/second
NAME                                 ID              SIZE     PROCESSOR    CONTEXT    RUNNER      UNTIL   
qwen3.8-27b-uncensored-mtp:latest    0cb38e1ce65b    19 GB    100% GPU     65536      llamacpp    Forever    


CompletedProcess(args=['ollama', 'ps'], returncode=0)

## 5. Expose the Ollama API with Cloudflare Quick Tunnel

**Run this cell in Kaggle only when you want remote access.**

Cloudflare Tunnel provides a public HTTPS address and forwards incoming requests to the Ollama service running at `http://127.0.0.1:11434` inside Kaggle.

When the cell succeeds, it prints three values:

- `PUBLIC_OLLAMA_URL` — the temporary public HTTPS URL.
- `PUBLIC_OLLAMA_URL/v1` — the OpenAI-compatible API base URL.
- `MTP_MODEL` — the model name clients must specify in requests.

### Why a tunnel is needed

The Ollama server is listening on a loopback address inside Kaggle. Other computers on the internet cannot directly reach that private address. The tunnel provides a public HTTPS entry point and forwards traffic to the local Ollama service.

### Security warning — read before running

This Cloudflare Quick Tunnel setup has **no API authentication**. Anyone who obtains the URL may be able to send prompts, consume your GPU time, and use the running model. Do not publish the URL in notebook outputs, screenshots, public repositories, or messages. Do not send confidential or personal data through it. Stop the tunnel and the Kaggle session when you finish.

**Implementation note:** The Cloudflare Tunnel code in the next cell is intentionally preserved exactly from the supplied reference notebook. Do not replace it with another tunneling method.

In [5]:
import queue
import re
import threading

def pipe_process_output(process, output_queue):
    for line in iter(process.stdout.readline, ""):
        output_queue.put(line)

tunnel_is_running = (
    "cloudflared_process" in globals()
    and cloudflared_process.poll() is None
    and "PUBLIC_OLLAMA_URL" in globals()
)

if not tunnel_is_running:
    cloudflared_process = subprocess.Popen(
        [
            "cloudflared", "tunnel",
            "--no-autoupdate",
            "--protocol", "http2",
            "--edge-ip-version", "4",
            "--url", OLLAMA_URL,
            "--http-host-header", "localhost:11434",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    output_queue = queue.Queue()
    threading.Thread(
        target=pipe_process_output,
        args=(cloudflared_process, output_queue),
        daemon=True,
    ).start()

    deadline = time.monotonic() + 45
    PUBLIC_OLLAMA_URL = None

    while time.monotonic() < deadline:
        if cloudflared_process.poll() is not None and output_queue.empty():
            break
        try:
            line = output_queue.get(timeout=1)
        except queue.Empty:
            continue

        match = re.search(
            r"https://[a-z0-9-]+\.trycloudflare\.com",
            line,
        )
        if match:
            PUBLIC_OLLAMA_URL = match.group(0)
            break

    if not PUBLIC_OLLAMA_URL:
        raise RuntimeError(
            "Cloudflare tunnel did not produce a public URL. "
            "Rerun this cell to try again."
        )

print("Public Ollama API:", PUBLIC_OLLAMA_URL)
print("OpenAI-compatible base URL:", f"{PUBLIC_OLLAMA_URL}/v1")
print("Model name:", MTP_MODEL)

Public Ollama API: https://withdrawal-tubes-pressing-geometry.trycloudflare.com
OpenAI-compatible base URL: https://withdrawal-tubes-pressing-geometry.trycloudflare.com/v1
Model name: qwen3.8-27b-uncensored-mtp


## 6. Use the model from another computer or application

After the tunnel cell prints the public URL, copy that URL into a client running on your **local computer** or another machine. Keep the Kaggle notebook session and tunnel running while sending requests.

You can interact with the same model in two ways:

- **Ollama native API:** use `/api/chat` for chat messages or `/api/generate` for a prompt-completion request.
- **OpenAI-compatible API:** use the `/v1` base URL with a client library that supports the OpenAI chat-completions interface.

The examples below are client-side examples. Copy the one you need into a local `.py` file or run it in your local Python environment. Replace `https://YOUR-TUNNEL.trycloudflare.com` with the exact URL printed by the Kaggle notebook.

### Option A — Call Ollama's native `/api/chat` endpoint

**Run this example on your local computer, not in the Kaggle notebook.**

Install the dependency if needed:

```bash
pip install requests
```

The request sends a user message to Ollama and asks for a non-streaming response. The response content is returned in `message.content`.

Replace the placeholder URL with your current `PUBLIC_OLLAMA_URL`. The model name must match the name printed by the notebook.

### Option B — Use the OpenAI-compatible `/v1` endpoint

Ollama provides an OpenAI-compatible API interface. This lets applications that support the OpenAI Python client communicate with the Ollama model by changing the **base URL**.

Install the client on your local computer:

```bash
pip install openai
```

The base URL must end in `/v1`, and the `model` value must be the Ollama model name printed by the notebook. In this local Ollama setup, the example uses the placeholder API key `ollama`; it is not an OpenAI account key.

**Run the following example locally, not in Kaggle.**

```python
# Run this example on your LOCAL computer, not in the Kaggle notebook.

from openai import OpenAI

PUBLIC_OLLAMA_URL = "https://YOUR-TUNNEL.trycloudflare.com"
MODEL_NAME = "qwen3.8-27b-uncensored-mtp"

client = OpenAI(
    base_url=f"{PUBLIC_OLLAMA_URL}/v1",
    api_key="ollama",  # Ollama does not require a real OpenAI API key here.
)

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Write a Python function to check a palindrome."}
    ],
)

print(response.choices[0].message.content)
```


## 7. Use the helper function inside Kaggle

This optional helper function sends chat requests to the Ollama server. By default, it calls the local Ollama address from inside Kaggle.

If the tunnel is running, you can also pass `PUBLIC_OLLAMA_URL` as `base_url` to send a request through the public HTTPS route. This is useful for a quick connectivity check, although your local computer examples above demonstrate the more common remote-client workflow.

In [8]:
def chat(prompt, base_url=OLLAMA_URL, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = ollama_request(
        "/api/chat",
        {
            "model": MTP_MODEL,
            "messages": messages,
            "stream": False,
            "keep_alive": -1,
            "options": {
                "draft_num_predict": DRAFT_TOKENS,
                "num_ctx": NUM_CTX,
            },
        },
        base_url=base_url,
    )
    return response["message"]["content"]

# Example local request:
print(chat("Write a concise hello-world program in Python."))

# Example request through the public tunnel:
print(chat("Hello from the tunnel.", base_url=PUBLIC_OLLAMA_URL))

```python
print("Hello, World!")
```
Hello back!

Can you see the light at the end yet?

Is this a literal tunnel (like a subway, a cave, or a hike), or are you navigating a metaphorical one (a difficult time, a period of intense focus, or a creative process)?

How is the air quality down there?


In [7]:
# !ollama run qwen3.8-27b-uncensored-mtp